# femtosecond PINN — Colab runner

Thin launcher only (GUIDE.md §4.6). No model or physics code lives here.

1. Runtime → Change runtime type → **GPU**
2. Add a Colab **Secret** named `GITHUB_TOKEN` (fine-grained PAT with read access to the private repo)
3. Run all cells

In [ ]:
# 1. GPU + Drive
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone / pull the repo (token from Colab Secrets, never pasted here)
import os, subprocess
from google.colab import userdata

REPO = 'leejoohyunn/pinn-femtosecond-laser-ablation'
WORK = '/content/femtosecond'
token = userdata.get('GITHUB_TOKEN')
url = f'https://{token}@github.com/{REPO}.git'

if os.path.isdir(WORK):
    subprocess.run(['git', '-C', WORK, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', url, WORK], check=True)
    subprocess.run(['git', '-C', WORK, 'remote', 'set-url', 'origin', f'https://github.com/{REPO}.git'], check=True)  # drop token from .git/config
%cd {WORK}
!git log --oneline -1

In [ ]:
# 3. Install (torch is preinstalled on Colab; deepxde version pinned in requirements.txt)
%pip install -q -e . -r requirements.txt
%env DDE_BACKEND=pytorch

In [ ]:
# 4. outputs/ → Google Drive so results survive disconnects
import os
DRIVE_OUT = '/content/drive/MyDrive/femtosecond/outputs'
os.makedirs(DRIVE_OUT, exist_ok=True)
if os.path.islink('outputs'):
    os.unlink('outputs')
elif os.path.isdir('outputs'):
    import shutil; shutil.rmtree('outputs')
os.symlink(DRIVE_OUT, 'outputs')
!ls -la outputs

In [ ]:
# 5. Phase 0 smoke check (DoD: glass n_cr 1.83e27 m⁻³, device = cuda)
!python scripts/check_env.py
!pytest -q

## Launch commands (later Phases)

Same commands as on the Mac, e.g.

```
!python scripts/train_forward.py --config configs/forward_glass.yaml --profile full
!python scripts/train_forward.py --config configs/forward_glass.yaml --profile full --resume outputs/forward/<run_dir>
```